**6 Training Jobs for HPO**

In [13]:
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.tuner import IntegerParameter, ContinuousParameter, HyperparameterTuner

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sess.default_bucket()

# 1. POINT TO YOUR SINGLE CLEANED FILE IN S3
s3_input_uri = "s3://churn-pred-umair/cleaned_data/churn_data_cleaned.csv"

# 2. DEFINE THE VAULT (Where weights will be stored)
# This ensures NOTHING is stored in your local "shared" directory
s3_output_location = "s3://churn-pred-umair/model_weights/output"

# 3. DEFINE ESTIMATOR
sklearn_estimator = SKLearn(
    entry_point='train.py',
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge', 
    framework_version='1.2-1',
    py_version='py3',
    sagemaker_session=sess,
    output_path=s3_output_location  
)

# 4. CONFIGURE HPO
tuner = HyperparameterTuner(
    sklearn_estimator,
    objective_metric_name='auc',
    metric_definitions=[{'Name': 'auc', 'Regex': 'VALIDATION_AUC: ([0-9\\.]+)'}],
    hyperparameter_ranges={
        'n_estimators': IntegerParameter(50, 150),
        'learning_rate': ContinuousParameter(0.01, 0.2)
    },
    max_jobs=6,
    max_parallel_jobs=2,
    strategy='Bayesian'
)

# 5. START
tuner.fit({'train': s3_input_uri})

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
...........................................................!


**Single Training Job**

In [21]:
# 1. Update your SKLearn Estimator with the BEST parameters
# (Replace 100 and 0.1 with your actual winning values)
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.tuner import IntegerParameter, ContinuousParameter, HyperparameterTuner

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sess.default_bucket()

# 1. POINT TO YOUR SINGLE CLEANED FILE IN S3
s3_input_uri = "s3://churn-pred-umair/cleaned_data/churn_data_cleaned.csv"

# 2. DEFINE THE VAULT (Where weights will be stored)
# This ensures NOTHING is stored in your local "shared" directory
s3_output_location = "s3://churn-pred-umair/model_weights"

final_estimator = SKLearn(
    entry_point='train.py',
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge', 
    framework_version='1.2-1',
    py_version='py3',
    sagemaker_session=sess,
    output_path="s3://churn-pred-umair/model_weights",
    hyperparameters={
        'n_estimators': 100,      # Your HPO Winner value
        'learning_rate': 0.1      # Your HPO Winner value
    }
)

# 2. Start the Single Training Job
# This will create a NEW model.tar.gz that includes your model_fn
final_estimator.fit({'train': s3_input_uri})

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
2026-03-19 18:12:26 Starting - Starting the training job...
2026-03-19 18:12:40 Starting - Preparing the instances for training...
2026-03-19 18:13:23 Downloading - Downloading the training image......
2026-03-19 18:14:09 Training - Training image download completed. Training in progress./miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pk

**Find Best Training Job**

In [ ]:
# 1. Get the name of the winning training job
best_job_name = tuner.best_training_job()

# 2. Get the full S3 path to that winner's weights
# This solves your "long messy path" problem by finding it for you
best_model_s3 = sess.sagemaker_client.describe_training_job(
    TrainingJobName=best_job_name
)['ModelArtifacts']['S3ModelArtifacts']

print(f"🏆 The Winning Job: {best_job_name}")
print(f"📦 Best Model S3 Path: {best_model_s3}")